# VISTA imputation: Huuki snRNA-seq → Xenium Br8667

This notebook uses:

- **Reference:** Huuki snRNA-seq Br8667
- **Spatial target:** Xenium Br8667
- **Expression input:** raw counts already stored in `.X`
- **Spatial coordinates:** existing `spatial_data.obsm["spatial"]`

The old `x_coord` / `y_coord` demo-specific code has been removed.

In [1]:
from pathlib import Path
import random

import anndata as ad
import matplotlib as mpl
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as st
from scipy import sparse
import torch

from vista import GIMVI_GCN

SEED = 2023
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/scratch/mjabin/conda_envs/Imputation/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Global seed set to 0


PyTorch: 2.1.2+cu118
CUDA available: False


/scratch/mjabin/conda_envs/Imputation/lib/python3.9/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
def _mean_expression(adata):
    """Return mean expression across observations as a one-dimensional array."""
    if sparse.issparse(adata.X):
        return np.asarray(adata.X.mean(axis=0)).ravel()
    return np.asarray(adata.X).mean(axis=0)


def calculate_pseudo_correlation(
    adata_sc,
    adata_st,
    celltype="scClassify",
    p_value_threshold=0.05,
    cor_threshold=0.5,
    exclude_types=("Other",),
):
    """Select measured spatial genes with consistent cell-type pseudobulk patterns."""

    if celltype not in adata_sc.obs:
        raise KeyError(f"{celltype!r} is missing from snRNA obs.")
    if celltype not in adata_st.obs:
        raise KeyError(f"{celltype!r} is missing from Xenium obs.")

    overlap_genes = sorted(set(adata_sc.var_names) & set(adata_st.var_names))
    if not overlap_genes:
        raise ValueError("No genes overlap between snRNA and Xenium.")

    sc_overlap = adata_sc[:, overlap_genes]
    st_overlap = adata_st[:, overlap_genes]

    sc_types = set(sc_overlap.obs[celltype].astype(str))
    st_types = set(st_overlap.obs[celltype].astype(str))
    common_types = sorted((sc_types & st_types) - set(exclude_types))

    print("Common cell types used:", common_types)

    if len(common_types) < 3:
        raise ValueError(
            "At least three common cell types are needed for Pearson correlation. "
            f"Found: {common_types}"
        )

    pseudo_sc = []
    pseudo_st = []

    for label in common_types:
        sc_subset = sc_overlap[sc_overlap.obs[celltype].astype(str) == label]
        st_subset = st_overlap[st_overlap.obs[celltype].astype(str) == label]

        if sc_subset.n_obs == 0 or st_subset.n_obs == 0:
            continue

        pseudo_sc.append(_mean_expression(sc_subset))
        pseudo_st.append(_mean_expression(st_subset))

    pseudo_sc = np.asarray(pseudo_sc)
    pseudo_st = np.asarray(pseudo_st)

    records = []
    for gene_index, gene in enumerate(overlap_genes):
        x = pseudo_sc[:, gene_index]
        y = pseudo_st[:, gene_index]

        if np.allclose(x, x[0]) or np.allclose(y, y[0]):
            correlation, pvalue = np.nan, np.nan
        else:
            correlation, pvalue = st.pearsonr(x, y)

        records.append((gene, correlation, pvalue))

    information = pd.DataFrame(
        records,
        columns=["gene", "pearson", "pvalue"],
    ).set_index("gene")

    selected = information.loc[
        (information["pvalue"] < p_value_threshold)
        & (information["pearson"] > cor_threshold)
    ]

    print("Measured overlapping genes:", len(overlap_genes))
    print("Genes retained for Xenium training:", selected.shape[0])

    return selected, information

In [3]:
PROJECT_ROOT = Path("/users/mjabin/projects/GeneBridge")
BASE = PROJECT_ROOT / "data/processed/imputation_beta/Br8667"
REPORT_DIR = PROJECT_ROOT / "outputs/imputation_beta/Br8667"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_PATH = BASE / "seq_data_huuki_snrna_Br8667_vista.h5ad"
SPATIAL_PATH = BASE / "spatial_data_xenium_Br8667_vista.h5ad"

print("snRNA:", SEQ_PATH)
print("Xenium:", SPATIAL_PATH)

seq_data = sc.read_h5ad(SEQ_PATH)
spatial_data = sc.read_h5ad(SPATIAL_PATH)

print("\nseq_data:")
print(seq_data)

print("\nspatial_data:")
print(spatial_data)

snRNA: /users/mjabin/projects/GeneBridge/data/processed/imputation_beta/Br8667/seq_data_huuki_snrna_Br8667_vista.h5ad
Xenium: /users/mjabin/projects/GeneBridge/data/processed/imputation_beta/Br8667/spatial_data_xenium_Br8667_vista.h5ad

seq_data:
AnnData object with n_obs × n_vars = 9897 × 28907
    obs: 'Sample', 'Barcode', 'key', 'SAMPLE_ID', 'pos', 'BrNum', 'round', 'Position', 'age', 'sex', 'diagnosis', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'high_mito', 'low_sum', 'low_detected', 'discard_auto', 'doubletScore', 'prelimCluster', 'collapsedCluster', 'kmeans', 'sizeFactor', 'cellType_broad_k', 'cellType_k', 'cellType_broad_hc', 'cellType_hc', 'cellType_layer', 'layer_annotation', 'scClassify_original', 'scClassify', 'names', 'batch'
    var: 'source', 'type', 'gene_id', 'gene_version', 'gene_name', 'gene_type', 'binomial_deviance', 'matched_gene_symbol', 'original_var_name', 'vista_gene_symbol'
    uns: 'Samples', 'X_name', 'c

In [4]:
# Validate metadata and use the spatial coordinates already present in the Xenium file.

required_seq_obs = {"scClassify", "names"}
required_spatial_obs = {"scClassify", "names", "batch"}

missing_seq = required_seq_obs - set(seq_data.obs.columns)
missing_spatial = required_spatial_obs - set(spatial_data.obs.columns)

if missing_seq:
    raise KeyError(f"Missing snRNA obs columns: {sorted(missing_seq)}")
if missing_spatial:
    raise KeyError(f"Missing Xenium obs columns: {sorted(missing_spatial)}")
if "spatial" not in spatial_data.obsm:
    raise KeyError(
        "Missing spatial_data.obsm['spatial']. "
        f"Available obsm keys: {list(spatial_data.obsm.keys())}"
    )

seq_data.obs["ind_x"] = seq_data.obs_names.astype(str)
spatial_data.obs["ind_x"] = spatial_data.obs_names.astype(str)

spatial_coordinates = np.asarray(
    spatial_data.obsm["spatial"],
    dtype=np.float32,
)

if spatial_coordinates.shape != (spatial_data.n_obs, 2):
    raise ValueError(
        f"Expected Xenium coordinates with shape ({spatial_data.n_obs}, 2), "
        f"received {spatial_coordinates.shape}."
    )

spatial_data.obsm["spatial"] = spatial_coordinates

print("Xenium genes are a subset of snRNA genes:",
      set(spatial_data.var_names) <= set(seq_data.var_names))
print("Spatial shape:", spatial_data.obsm["spatial"].shape)
print("First five coordinates:")
print(spatial_data.obsm["spatial"][:5])

Xenium genes are a subset of snRNA genes: True
Spatial shape: (66164, 2)
First five coordinates:
[[ 174.27339 1633.0416 ]
 [ 199.3187  1638.3245 ]
 [ 196.35602 1625.7012 ]
 [ 187.38412 1634.797  ]
 [ 211.25597 1633.4785 ]]


In [5]:
print("snRNA scClassify:")
print(seq_data.obs["scClassify"].astype(str).value_counts())

print("\nXenium scClassify:")
print(spatial_data.obs["scClassify"].astype(str).value_counts())

common_types = sorted(
    set(seq_data.obs["scClassify"].astype(str))
    & set(spatial_data.obs["scClassify"].astype(str))
)

print("\nCommon scClassify labels:")
print(common_types)

snRNA scClassify:
Oligo        5305
Excit        2613
Other        1214
Inhib         340
Astro         250
EndoMural     114
OPC            61
Name: scClassify, dtype: int64

Xenium scClassify:
Oligo        14701
EndoMural    13821
Excit        13268
Astro         9247
Inhib         6702
Micro         4577
Other         3848
Name: scClassify, dtype: int64

Common scClassify labels:
['Astro', 'EndoMural', 'Excit', 'Inhib', 'Oligo', 'Other']


In [6]:
selected_gene_table, all_gene_statistics = calculate_pseudo_correlation(
    seq_data,
    spatial_data,
    celltype="scClassify",
    p_value_threshold=0.05,
    cor_threshold=0.5,
    exclude_types=("Other",),
)

all_gene_statistics.to_csv(
    REPORT_DIR / "vista_Br8667_pseudocorrelation_all_300_genes.csv"
)
selected_gene_table.to_csv(
    REPORT_DIR / "vista_Br8667_pseudocorrelation_selected_genes.csv"
)

info_genes = selected_gene_table.index.tolist()

if len(info_genes) < 10:
    raise ValueError(
        f"Only {len(info_genes)} genes passed pseudocorrelation filtering. "
        "Review the cell-type mapping or relax the thresholds deliberately."
    )

print("Selected genes:", len(info_genes))
print(info_genes[:20])

Common cell types used: ['Astro', 'EndoMural', 'Excit', 'Inhib', 'Oligo']
Measured overlapping genes: 300
Genes retained for Xenium training: 188
Selected genes: 188
['A2M', 'ABCG2', 'ACO1', 'ACTA2', 'ADCYAP1', 'ADCYAP1R1', 'ADIRF', 'AJ009632.2', 'AK5', 'AKAP8L', 'ALDH1A1', 'ANLN', 'APOLD1', 'AQP4', 'ARPC1B', 'ARPP19', 'ARRDC3', 'ATP2B4', 'BAIAP3', 'BCAS1']


In [7]:
# The complete snRNA transcriptome is the imputation target.
gene_for_impute = seq_data.var_names.copy()

# Xenium is trained only on informative measured genes.
seq_data_model = seq_data[:, gene_for_impute].copy()
spatial_data_model = spatial_data[:, info_genes].copy()

print("snRNA model input:", seq_data_model.shape)
print("Xenium model input:", spatial_data_model.shape)
print(
    "Xenium model genes subset of snRNA:",
    set(spatial_data_model.var_names) <= set(seq_data_model.var_names),
)

snRNA model input: (9897, 28907)
Xenium model input: (66164, 188)
Xenium model genes subset of snRNA: True


In [8]:
# Register AnnData objects for VISTA.
# The prepared files already contain raw counts in X.

GIMVI_GCN.setup_anndata(
    spatial_data_model,
    batch_key="batch",
    obs_names="names",
)

GIMVI_GCN.setup_anndata(seq_data_model)

print("VISTA AnnData setup complete.")

VISTA AnnData setup complete.


/scratch/mjabin/conda_envs/Imputation/lib/python3.9/site-packages/scvi/data/_utils.py:172: UserWarning: Category 0 in adata.obs['_scvi_ind_x'] has fewer than 3 cells. Models may not train properly.
  warnings.warn(


In [9]:
model = GIMVI_GCN(
    seq_data_model,
    spatial_data_model,
    n_latent=32,
    neighbor_size=20,
)

print(model)

/scratch/mjabin/conda_envs/Imputation/lib/python3.9/site-packages/scvi/model/_utils.py:286: UserWarning: This dataset has some empty cells, this might fail inference.Data should be filtered with `scanpy.pp.filter_cells()`
  warnings.warn(


GimVI Model with the following params: 
n_latent: 32, n_inputs: [28907, 188], n_genes: 28907, n_batch: 2, generative distributions: ['zinb', 'nb']
Training status: Not Trained
Model's adata is minified?: False

In [10]:
# # Train on the GPU.
# model.train(
#     max_epochs=200,
#     use_gpu=True,
#     batch_size=128,
# )

# print("Training complete.")


############################# Cell Split ######################################
# snRNA:
# 8,908 training nuclei
# 989 validation nuclei

# Xenium:
# 59,548 training cells
# 6,616 validation cells

model.train(
    max_epochs=2,
    train_size=0.9,
    validation_size=None,
    use_gpu=False,
    batch_size=128,
)

print("CPU smoke test complete.")

print("Training complete.")

/scratch/mjabin/conda_envs/Imputation/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /scratch/mjabin/conda_envs/Imputation/lib/python3.9/ ...
  rank_zero_warn(
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/scratch/mjabin/conda_envs/Imputation/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /scratch/mjabin/conda_envs/Imputation/lib/python3.9/ ...
  rank_zero_warn(


Epoch 2/2: 100%|██████████| 2/2 [02:30<00:00, 74.96s/it, loss=3.67e+03, v_num=1]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 2/2: 100%|██████████| 2/2 [02:30<00:00, 75.25s/it, loss=3.67e+03, v_num=1]
CPU smoke test complete.
Training complete.


In [11]:
# import matplotlib.pyplot as plt
# plt.plot(train_loss)
# plt.plot(val_loss)
# plt.yscale('log')

In [ ]:
# Save the trained model before producing the very large dense imputation matrix.

MODEL_DIR = REPORT_DIR / "vista_Br8667_model"

model.save(
    str(MODEL_DIR),
    overwrite=True,
    save_anndata=False,
)

print("Saved model:", MODEL_DIR)

Saved model: /users/mjabin/projects/GeneBridge/outputs/imputation_beta/Br8667/vista_Br8667_model


: 

## Optional: produce the full imputed matrix

The full matrix has approximately **66,164 Xenium cells × 28,907 genes**. A single
dense `float32` matrix is about 7.1 GiB before HDF5 overhead. Run the next cell
only when sufficient RAM and scratch storage are available.

In [ ]:
RUN_FULL_IMPUTATION = True

if RUN_FULL_IMPUTATION:
    imputed_raw = model.get_imputed_values(
        normalized=False,
        batch_size=128,
    )[0].astype(np.float32, copy=False)

    spatial_data_imputed = ad.AnnData(
        X=imputed_raw,
        obs=spatial_data_model.obs.copy(),
        var=seq_data_model.var.copy(),
    )

    spatial_data_imputed.obsm["spatial"] = (
        spatial_data_model.obsm["spatial"].copy()
    )

    IMPUTED_PATH = BASE / "vista_Br8667_imputed_raw.h5ad"
    spatial_data_imputed.write_h5ad(
        IMPUTED_PATH,
        compression="gzip",
    )

    print("Saved imputed matrix:", IMPUTED_PATH)
    print(spatial_data_imputed)
else:
    print(
        "Full imputation skipped. Set RUN_FULL_IMPUTATION = True "
        "after checking RAM and scratch capacity."
    )